# Data inspection and cleaning

**Project:** Friendship contact and loneliness in the EU Loneliness Survey 2022.

This notebook checks and prepares the data for our exploratory analysis.
Our workflow draws on the inspection and preprocessing methods covered in Weeks 3 and 4.

The main analysis uses respondents with a valid age of 16 or over, both friendship-contact responses and all three loneliness items. We also prepare contextual variables to explore wider social and personal circumstances.

We retain all respondents in a separate cleaned file, with flags showing eligibility and missingness. No imputation or survey weighting is applied at this stage.

Sources:
- [JRC dataset](https://doi.org/10.2905/JRC.V4VT8T8)
- [EU27 questionnaire](https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/EU-Loneliness-Survey/eu_loneliness_survey_quest_eu27.pdf)


## 1 Import libraries and locate files


In [ ]:
from pathlib import Path
import hashlib
import pandas as pd
from IPython.display import display

# Resolve paths from either the project root or notebooks folder.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
VALUES_NAME = "eu_loneliness_survey_eu27_values.csv"
LABELS_NAME = "eu_loneliness_survey_eu27_labels.csv"

VALUES_PATH = RAW_DIR / VALUES_NAME
LABELS_PATH = RAW_DIR / LABELS_NAME
for path in [VALUES_PATH, LABELS_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f"Cannot find {path}. Check RAW_DIR in this cell.")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "results" / "tables"
for folder in [PROCESSED_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Store hashes to check that the raw files are unchanged after cleaning.
raw_hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
              for p in [VALUES_PATH, LABELS_PATH]}
print("Raw data folder:", RAW_DIR)

## 2 Load and inspect the dataset

We check the dataset dimensions, variable types, respondent IDs, duplicates and initial value ranges. The values and labels files describe the same respondents, so we check their alignment rather than combining them as separate observations.


In [ ]:
raw = pd.read_csv(VALUES_PATH, low_memory=False)
labels = pd.read_csv(LABELS_PATH, low_memory=False)

contact_cols = ["friends_meet_face", "friends_meet_tele"]
ucla_cols = ["loneliness_ucla_a", "loneliness_ucla_b", "loneliness_ucla_c"]
core_cols = contact_cols + ucla_cols
weight_cols = ["w_country_5", "w_country", "w_population", "w_eu27_5", "w_eu27"]

context_cols = [
    "gender", "relationship", "work_status", "education", "municipality_type",
    "health_general", "health_condition",
    "social_support_a", "social_support_b", "social_support_c", "social_support_d",
    "family_meet_face", "family_meet_tele", "social_activities_a",
    "friends_n__1", "friends_n__1__open", "family_n__1", "family_n__1__open",
    "adults_n", "childr_n_0to5", "childr_n_6to15", "others_n",
]

selected_cols = ["id_pers", "country", "age"] + core_cols + context_cols + weight_cols

assert set(selected_cols).issubset(raw.columns), "A required column is missing."
assert raw.columns.equals(labels.columns), "The file columns differ."
assert raw["id_pers"].equals(labels["id_pers"]), "Respondents are not aligned."
assert raw["id_pers"].notna().all(), "Investigate missing respondent IDs."
assert raw["id_pers"].is_unique, "Investigate repeated IDs before proceeding."

print("Rows and columns:", raw.shape)
print("Countries:", raw["country"].nunique())
print("Exact duplicate rows:", raw.duplicated().sum())
display(raw[selected_cols].head())
raw[selected_cols].info()
display(raw[core_cols + ["age"]].describe())

## 3 Check response codes

We match the numerical codes to the supplied labels before cleaning. This identifies coded nonresponses that would not appear in a check for blank values.


In [ ]:
mapping_tables = []
for col in core_cols + ["age"] + context_cols:
    mapping = pd.DataFrame({"code": raw[col], "label": labels[col]})
    mapping = mapping.value_counts(dropna=False).reset_index(name="n")
    mapping.insert(0, "variable", col)
    mapping_tables.append(mapping)
    if col in core_cols:
        print(col)
        display(mapping.sort_values("code"))

codebook_observed = pd.concat(mapping_tables, ignore_index=True)
print("Unusual age codes:")
display(codebook_observed.loc[
    (codebook_observed["variable"] == "age") &
    ((codebook_observed["code"] < 16) | (codebook_observed["code"] == 997))
])

## 4 Clean core responses

We work on a copy of the selected variables. In the five core response fields, 999 means “prefer not to say” and is recoded as missing.

Unexpected codes are checked separately. We use variable-specific rules rather than replacing codes across the whole dataset.


In [ ]:
clean = raw[selected_cols].copy()
audit_rows = []

for col in core_cols:
    allowed = list(range(1, 8)) if col in contact_cols else [1, 2, 3]
    unexpected = clean[col].notna() & ~clean[col].isin(allowed + [999])
    if unexpected.any():
        raise ValueError(f"Unexpected codes in {col}: {clean.loc[unexpected, col].unique()}")
    audit_rows.append({
        "variable": col,
        "blank_before": int(clean[col].isna().sum()),
        "prefer_not_to_say": int(clean[col].eq(999).sum()),
    })
    clean[col] = clean[col].replace(999, float("nan"))

missing_audit = pd.DataFrame(audit_rows).set_index("variable")
missing_audit["missing_after"] = clean[core_cols].isna().sum()
missing_audit["missing_percent"] = (100 * clean[core_cols].isna().mean()).round(2)
display(missing_audit)

## 5 Check age eligibility

We preserve the original age values and recode 997, labelled as implausible, as missing. Recorded ages below 16 remain unchanged but are excluded from the main analysis.

Requiring a known age of at least 16 is our project decision to match the research question. All records remain in the cleaned file.


In [ ]:
clean["age_raw"] = clean["age"]
clean["age"] = clean["age"].replace(997, float("nan"))
clean["age_missing"] = clean["age"].isna()
clean["age_under_16"] = clean["age"].lt(16)
clean["age_eligible"] = clean["age"].notna() & clean["age"].ge(16)

age_audit = pd.DataFrame({
    "condition": ["Missing or implausible age", "Recorded age below 16", "Confirmed age 16+"],
    "n": [clean["age_missing"].sum(), clean["age_under_16"].sum(), clean["age_eligible"].sum()]
})
display(age_audit)

## 6 Prepare contextual variables

We prepare demographic, health, support and family-contact variables for the contextual analysis. Readable labels are added alongside the cleaned numerical codes.

Nominal codes, such as relationship status, are category identifiers rather than numerical ranks. Contact frequency, support and general health are ordered categories, but equal intervals are not assumed.

Higher health codes indicate poorer health, while higher support codes indicate more available support. The four support items are kept separate.


In [ ]:
# Valid responses and missing codes differ between variables.
category_rules = {
    "gender": ([1, 2, 3], [999]),
    "relationship": (list(range(1, 6)), [999]),
    "work_status": (list(range(1, 9)), [999]),
    "education": (list(range(1, 6)), [999]),
    "municipality_type": ([1, 2, 3], [998]),
    "health_general": (list(range(1, 6)), [998, 999]),
    "health_condition": ([1, 2], [998, 999]),
    "family_meet_face": (list(range(1, 8)), [999]),
    "family_meet_tele": (list(range(1, 8)), [999]),
    "social_activities_a": (list(range(1, 8)), [999]),
}
for col in ["social_support_a", "social_support_b", "social_support_c", "social_support_d"]:
    category_rules[col] = (list(range(1, 6)), [998, 999])

context_audit_rows = []
for col, (valid_codes, missing_codes) in category_rules.items():
    unexpected = clean[col].notna() & ~clean[col].isin(valid_codes + missing_codes)
    if unexpected.any():
        raise ValueError(f"Review unexpected codes in {col}: {clean.loc[unexpected, col].unique()}")
    context_audit_rows.append({
        "variable": col,
        "blank_before": int(clean[col].isna().sum()),
        "dont_know": int(clean[col].eq(998).sum()),
        "prefer_not_to_say": int(clean[col].eq(999).sum()),
    })
    clean[col] = clean[col].replace(missing_codes, float("nan"))
    mapping = pd.DataFrame({"code": raw[col], "label": labels[col]}).drop_duplicates()
    assert mapping["code"].is_unique, f"Ambiguous labels in {col}"
    clean[col + "_label"] = clean[col].map(mapping.set_index("code")["label"])

context_missing_audit = pd.DataFrame(context_audit_rows).set_index("variable")
context_missing_audit["missing_after"] = clean[list(category_rules)].isna().sum()
context_missing_audit["missing_percent"] = (100 * clean[list(category_rules)].isna().mean()).round(2)
display(context_missing_audit)

## 7 Check relationship and household counts

Close-friend and close-family counts are taken from the numeric-answer fields when the response flag confirms that an answer was supplied. A valid zero is retained; missing responses are not converted to zero.

Non-integer and out-of-range counts are flagged and treated as missing. Original values are preserved, and these checks do not remove respondents from the main sample.

We retain separate household counts rather than assuming that one adult means the respondent lives alone. Broad age and close-friend groups are created for subgroup exploration, alongside the original measurements.


In [ ]:
count_audit_rows = []
for prefix, new_col in [("friends", "close_friends_n"), ("family", "close_family_n")]:
    flag = clean[prefix + "_n__1"]
    count = clean[prefix + "_n__1__open"]
    assert flag.isin([1, 999]).all(), f"Unexpected response flag: {prefix}"
    assert count.loc[flag.eq(999)].isna().all(), f"Count present despite refusal: {prefix}"
    numeric_answer = flag.eq(1)
    valid_count = count.between(0, 100) & count.mod(1).eq(0)
    clean[new_col + "_invalid"] = numeric_answer & ~valid_count
    clean[new_col] = count.where(numeric_answer & valid_count)
    count_audit_rows.append({"variable": new_col, "valid_n": int(clean[new_col].notna().sum()),
                             "missing_n": int(clean[new_col].isna().sum()),
                             "invalid_n": int(clean[new_col + "_invalid"].sum()),
                             "zero_n": int(clean[new_col].eq(0).sum())})

household_ranges = {"adults_n": (1, 30), "childr_n_0to5": (0, 10),
                    "childr_n_6to15": (0, 20), "others_n": (0, 50)}
for col, (low, high) in household_ranges.items():
    present = clean[col].notna()
    valid = clean[col].between(low, high) & clean[col].mod(1).eq(0)
    clean[col + "_raw"] = clean[col]
    clean[col + "_invalid"] = present & ~valid
    clean[col] = clean[col].where(valid)
    count_audit_rows.append({"variable": col, "valid_n": int(clean[col].notna().sum()),
                             "missing_n": int(clean[col].isna().sum()),
                             "invalid_n": int(clean[col + "_invalid"].sum()),
                             "zero_n": int(clean[col].eq(0).sum())})
count_audit = pd.DataFrame(count_audit_rows)
display(count_audit)

# Create broad groups for subgroup plots; retain age and original counts.
clean["age_group"] = pd.cut(clean["age"], bins=[16, 30, 45, 60, float("inf")],
                            labels=["16-29", "30-44", "45-59", "60+"], right=False)
clean["close_friends_group"] = pd.cut(clean["close_friends_n"],
    bins=[-1, 0, 2, 5, 100], labels=["0", "1-2", "3-5", "6+"])

## 8 Calculate the loneliness score and select the main sample

The three UCLA items are scored from 1 to 3 in the same direction. Their sum ranges from 3 to 9, with higher scores indicating greater reported loneliness.

We require all three items for a valid total. The main sample also requires both contact responses and a valid age of at least 16.


In [ ]:
clean["ucla_items_answered"] = clean[ucla_cols].notna().sum(axis=1)
# min_count=3 prevents partial totals when an item is missing.
clean["ucla_total"] = clean[ucla_cols].sum(axis=1, min_count=3)
clean["core_complete"] = clean[core_cols].notna().all(axis=1)
clean["include_main_analysis"] = clean["age_eligible"] & clean["core_complete"]

analysis = clean.loc[clean["include_main_analysis"]].copy()
print("Rows with all five core responses:", int(clean["core_complete"].sum()))
print("Main analysis sample after age eligibility:", len(analysis))

## 9 Summarise the sample flow

We record exclusions in sequence so each respondent is counted once. Missing-response counts overlap across variables and cannot simply be added together to calculate the number excluded.

We also summarise the country composition of the final sample.


In [ ]:
n_raw = len(clean)
n_age = int(clean["age_eligible"].sum())
n_final = len(analysis)
sample_flow = pd.DataFrame({
    "stage": ["Original respondents", "Confirmed age 16+", "Complete core responses among eligible respondents"],
    "n_retained": [n_raw, n_age, n_final],
    "n_excluded_at_stage": [0, n_raw - n_age, n_age - n_final],
})
sample_flow["percent_of_original_retained"] = (100 * sample_flow["n_retained"] / n_raw).round(2)
display(sample_flow)

# Unweighted country counts in the main analysis sample.
country_counts = analysis["country"].value_counts().rename_axis("country").reset_index(name="n")
country_names = pd.DataFrame({"country": raw["country"], "country_name": labels["country"]}).drop_duplicates()
assert country_names["country"].is_unique
country_counts = country_counts.merge(country_names, on="country", how="left", validate="one_to_one")
display(country_counts)

## 10 Check contextual missingness and sample composition

We summarise valid and missing responses for the contextual variables within the main sample. Optional missingness does not change the core inclusion rule.

These tables document the data available for later comparisons, which use the valid responses for each relevant variable.


In [ ]:
context_analysis_cols = list(category_rules) + ["close_friends_n", "close_family_n"] + list(household_ranges)
context_availability = pd.DataFrame({
    "valid_n": analysis[context_analysis_cols].notna().sum(),
    "missing_n": analysis[context_analysis_cols].isna().sum(),
    "missing_percent": (100 * analysis[context_analysis_cols].isna().mean()).round(2),
})
context_availability.index.name = "variable"
display(context_availability)

composition_tables = []
for col in ["gender_label", "relationship_label", "work_status_label", "education_label",
            "municipality_type_label", "age_group", "close_friends_group"]:
    counts = analysis[col].astype("object").fillna("Missing / non-response").value_counts()
    table = counts.rename_axis("category").reset_index(name="n")
    table.insert(0, "variable", col)
    table["percent_of_main_sample"] = (100 * table["n"] / len(analysis)).round(2)
    composition_tables.append(table)
sample_composition = pd.concat(composition_tables, ignore_index=True)
display(sample_composition)

## 11 Validate and export

We check score ranges, completeness, identifiers and inclusion flags before exporting two datasets:

- A cleaned file retaining all respondents and flags.
- The complete-case sample used for the main analysis.

Audit tables are saved alongside the prepared data. The final hash check confirms that the raw files remain unchanged.


In [ ]:
assert len(clean) == len(raw)
assert analysis["id_pers"].is_unique
assert not analysis[core_cols + ["ucla_total", "age"]].isna().any().any()
assert analysis[contact_cols].isin(range(1, 8)).all().all()
assert analysis[ucla_cols].isin([1, 2, 3]).all().all()
assert analysis["ucla_total"].between(3, 9).all()
assert clean["ucla_total"].isna().equals(clean["ucla_items_answered"].lt(3))
assert analysis["age"].ge(16).all()

clean.to_csv(PROCESSED_DIR / "eu_loneliness_cleaned_with_flags.csv", index=False)
analysis.to_csv(PROCESSED_DIR / "eu_loneliness_analysis_sample.csv", index=False)
missing_audit.to_csv(TABLES_DIR / "missing_response_audit.csv")
age_audit.to_csv(TABLES_DIR / "age_audit.csv", index=False)
sample_flow.to_csv(TABLES_DIR / "sample_flow.csv", index=False)
codebook_observed.to_csv(TABLES_DIR / "observed_code_labels.csv", index=False)
country_counts.to_csv(TABLES_DIR / "analysis_country_counts.csv", index=False)

# Raw-sample audits and main-sample summaries are exported separately.
context_missing_audit.to_csv(TABLES_DIR / "context_missing_audit_raw_sample.csv")
count_audit.to_csv(TABLES_DIR / "close_relationship_count_audit_raw_sample.csv", index=False)
context_availability.to_csv(TABLES_DIR / "context_availability_main_sample.csv")
sample_composition.to_csv(TABLES_DIR / "sample_composition.csv", index=False)

assert analysis["id_pers"].equals(clean.loc[clean["age_eligible"] & clean["core_complete"], "id_pers"])
assert analysis["age_group"].notna().all()
assert analysis["close_friends_group"].isna().equals(analysis["close_friends_n"].isna())

for path in [VALUES_PATH, LABELS_PATH]:
    assert hashlib.sha256(path.read_bytes()).hexdigest() == raw_hashes[path.name]
print("Validation passed. Raw files are unchanged.")
print("Prepared datasets:", PROCESSED_DIR)
print("Audit tables:", TABLES_DIR)

## 12 Summary

The main analysis sample contains 24,431 respondents with valid age and complete core responses. The cleaned file retains all 25,646 original respondents, with flags documenting the inclusion decisions.

We have not imputed responses or applied survey weights. Complete-case selection may introduce bias if excluded respondents differ from those retained.

The prepared data are used in `02_exploratory_data_analysis.ipynb` for descriptive statistics, contact–loneliness comparisons and contextual exploration.
